In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd

import statsmodels.api as sm

from ISLP.models import (ModelSpec as MS, summarize , poly)
from ISLP import confusion_table

# Mixture Models
Sample from a mixture of three Gaussians:
$$
\mu = \pi_1 N(m_1, \sigma_1^2) + \pi_2 N(m_2, \sigma_2^2) + \pi_3 N(m_3, \sigma_3^2) 
$$
First generate a class (1,2, or 3), then generate a sample from the corresponding normal.

In [ ]:
rng = np.random.default_rng(318)

# Mixture model parameters
n = 10000
weights = np.array([0.3, 0.6, .1])          # mixing probabilities (sum to 1)
means   = np.array([-4.0,  1.0, 6.0])          # component means
sds     = np.array([ 2,  1, 0.5])          # component standard deviations

# 1) sample component labels z in {0,1,2}
# these are the class features
z = rng.choice(len(weights), size=n, p=weights)

# 2) sample x | z ~ Normal(mean[z], sd[z])
x = rng.normal(loc=means[z], scale=sds[z], size=n)

df = pd.DataFrame({"x": x, "z": z})
df.z = df.z.astype('category')
df.head()


In [ ]:
fig, ax = plt.subplots()

df.hist('x', bins=50, density=True, alpha=0.5, ax=ax)
ax.set_title('')
ax.set_xlabel(r'$x$')
ax.set_ylim(0, 0.5)

In [ ]:
fig, ax = plt.subplots()

for (z_val, group) in df.groupby('z'):
    group.hist('x', bins=20, density=True, alpha=0.5, ax=ax, label=f'z={z_val}')
ax.legend()
ax.set_title('')
ax.set_xlabel(r'$x$')
ax.set_ylim(0, 0.8)


# LDA

## Load in Data

In [ ]:
default = pd.read_csv("../data/Default.csv")
default['student'] = default['student'].astype('category')
default['default'] = default['default'].astype('category')
default.head()

In [ ]:
# set seed for reproducibility
train_df = default.sample(frac=0.7, random_state=123)   # 70% train
test_df  = default.drop(train_df.index)                 # 30% test

design = MS(['balance'], intercept=False) # intercept already handled in LDA/QDA
X_train = design.fit_transform(train_df)

X_test = design.transform(test_df)


## Load Module

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

## Fit

In [ ]:
lda = LDA(store_covariance=True)
lda.fit(X_train, train_df['default'])

## Check Performance

In [ ]:
pred_train = lda.predict(X_train)
pred_test = lda.predict(X_test)

print("Train error rate:", np.mean(pred_train != train_df['default']))
print("Test error rate:", np.mean(pred_test != test_df['default']))